In [3]:
import langchain
import openai
import dotenv

from langchain_openai import ChatOpenAI

import os

dotenv.load_dotenv() #加载当前目录下的.env文件

os.environ['DEEPSEEK_API_KEY'] = os.getenv("DEEPSEEK_API_KEY")
os.environ['DEEPSEEK_BASE_URL'] = os.getenv("DEEPSEEK_BASE_URL")

In [3]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "你是谁，能做什么？"},
    ],
    stream=True,
)

for chunk in completion:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。

具体来说，我可以帮助你：

✅ **学习与教育**：解释知识点、解数学题、写作文、翻译多国语言、辅助备考（如英语四六级、考研、托福雅思等）  
✅ **工作与办公**：撰写和润色邮件、报告、总结、PPT大纲；生成会议纪要；模拟面试问答；优化简历  
✅ **创意与内容创作**：写小说、诗歌、剧本、广告文案；起标题、写 slogan；设计角色设定与世界观  
✅ **技术与编程**：理解并生成多种编程语言（Python、Java、C++、JavaScript 等）；调试建议；写注释和文档；算法讲解  
✅ **生活与日常**：制定旅行计划、健身/学习计划；提供健康饮食建议（非医疗诊断）；推荐书籍/电影；陪你聊天、解压、头脑风暴  

需要注意的是：  
⚠️ 我不接入实时互联网（除非你使用“联网搜索”功能，且该功能需用户主动开启），因此无法获取2024年10月之后的最新资讯；  
⚠️ 我不能执行物理操作（如打开文件、控制设备），也不具备个人情感或主观意识；  
⚠️ 对于医疗、法律、金融等专业领域的问题，我会尽力提供通用信息和参考，但**不能替代专业人士的诊断、建议或服务**。

如果你有任何具体需求——比如“帮我写一封辞职信”“用Python画一个动态心形图”“解释量子纠缠是什么”——欢迎随时告诉我，我很乐意为你效劳！😊

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.tongyi import Tongyi
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.runnables import RunnableLambda

model = Tongyi(model="qwen-max")
str_parser = StrOutputParser()

first_prompt = PromptTemplate.from_template(
    "我邻居姓：{lastname}, 他的生了{gender},请帮忙起一个名字。只需要告诉我他的名字是什么，不需要其他的信息"
)

second_prompt = PromptTemplate.from_template(
    "姓名{name}, 请帮我解析含义"
)

my_func = RunnableLambda(lambda ai_msg: {"name": ai_msg.strip()})
chain = first_prompt | model | my_func | second_prompt | model | str_parser

for chunk in chain.stream({"lastname": "张", "gender":"女孩"}):
    print(chunk,end="",flush=True)


名字“张家欣”由三个汉字组成，每个字都有其独特的含义。我们可以逐一解析：

1. **张**：这是一个常见的中文姓氏之一，在中国有着悠久的历史。据《百家姓》记载，“张”姓源自姬姓，是周文王的后代。“张”字本身也有展开、扩大等意思。

2. **家**：这个字在中文里代表着家庭或家族的意思，也可以指代一个群体或者某人的住所。它传达了一种归属感和安全感，同时也暗示着团结与和谐。

3. **欣**：意为快乐、高兴、满意等正面情绪。用作人名时，通常寄托了父母希望孩子能够一生幸福快乐的美好愿望。

综上所述，“张家欣”这个名字整体上传达了一个温馨而积极的信息——属于张家的一员，并且希望这个人能够给家庭带来喜悦，自己也能够生活得非常开心满足。当然，每个人对自己名字的理解可能会有所不同，最重要的是这个名字背后所蕴含的家庭情感和个人故事。

In [10]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

model = ChatTongyi(model="qwen3-max")
prompt = PromptTemplate.from_template(
    "你需要根据历史记录回答用户的问题，对话历史：{chat_history}，用户提问：{input}"
)

str_parser = StrOutputParser()

base_chain = prompt | model | str_parser


store = {}
def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation_chain = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

session_config = {
    "configurable":{
        "session_id": "user_001"
    }
}
conversation_chain.invoke({"input":"小明有 2 只猫"},config=session_config)
conversation_chain.invoke({"input":"这两只猫分别生了 1 只猫"},config=session_config)
res = conversation_chain.invoke({"input":"现在一共有多少只猫？"},config=session_config)

res

'根据之前的对话：\n\n- 小明原本有 2 只猫。  \n- 这两只猫分别生了 1 只小猫，也就是一共新生了 2 只小猫。\n\n所以现在小明一共有：  \n2（原来的猫） + 2（新生的小猫） = **4 只猫**。\n\n前提是这两只猫都是母猫并且各自生了一只小猫。在这一假设下，答案是 **4 只猫**。'

AttributeError: 'TextAccessor' object has no attribute 'to_string'